# 07 — Session 与 CLI：把 Agent 包装成可用工具

前六章造好了所有核心模块：

```
LLMClient         ← 和 Ollama 通信
ContextManager    ← 管对话历史和 token
ToolRegistry      ← 工具注册和调用
AgenticLoop       ← 主循环，驱动 LLM + 工具自主运转
```

这章把它们组装成两样东西：

1. **Session** — 一次对话的完整上下文容器（不是单条消息，是整个会话）
2. **CLI** — 命令行界面，包括 Rich 终端 UI 和 Click 命令入口

```
用户输入
   │
   ▼
┌──────────────────────────────────────┐
│  CLI (Click 入口 + Rich UI 渲染)      │
│                                      │
│  ┌─────────────────────────────────┐ │
│  │  Session                        │ │
│  │  ├── LLMClient                  │ │
│  │  ├── ContextManager             │ │
│  │  ├── ToolRegistry               │ │
│  │  └── turn_count / metadata      │ │
│  └─────────────────────────────────┘ │
│                                      │
│  run_interactive() / --prompt 模式   │
└──────────────────────────────────────┘
```

---

**依赖安装**：
```bash
pip install rich click
```

In [ ]:
import sys
sys.path.insert(0, "..")

# 确认依赖可用
import rich
import click
print(f"rich {rich.__version__}")
print(f"click {click.__version__}")

## 1. Session 类

### 为什么要有 Session

Agent 运行期间需要同时持有三个对象：`LLMClient`、`ContextManager`、`ToolRegistry`。
每次调用函数都要把这三个传来传去，代码很丑。

Session 把它们包成一个容器，附加会话元数据：
- `session_id`：每次启动生成唯一 ID，方便日志追踪
- `turn_count`：记录交互轮次，`/stats` 命令会用到
- `created_at / updated_at`：时间戳，方便查看会话时长

### 初始化逻辑

`initialize()` 负责创建所有子模块。把初始化独立出来（而不是放在 `__init__`），
是因为 `LLMClient` 的连接是异步的，Python 的 `__init__` 不能是 `async`。

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from uuid import uuid4

from src.llm_client import LLMClient
from src.context_manager import ContextManager, build_system_prompt
from src.tool_framework import ToolRegistry


@dataclass
class SessionConfig:
    """Session 运行时参数，统一管理，方便 CLI 传入。"""
    model: str = "gpt-oss:120b"
    base_url: str = "http://localhost:11434/v1"
    api_key: str = "ollama"
    max_turns: int = 10        # 单次对话最多跑几轮 agentic loop
    temperature: float = 0.7
    max_tokens: int = 4096
    cwd: Path = field(default_factory=lambda: Path("."))


@dataclass
class Session:
    """一次 Agent 会话的完整上下文容器。"""
    session_id: str = field(default_factory=lambda: str(uuid4()))
    created_at: datetime = field(default_factory=datetime.now)
    updated_at: datetime = field(default_factory=datetime.now)
    turn_count: int = 0
    config: SessionConfig = field(default_factory=SessionConfig)

    # 子模块，initialize() 后才可用
    context_manager: ContextManager = field(default=None, repr=False)
    tool_registry: ToolRegistry = field(default=None, repr=False)
    llm_client: LLMClient = field(default=None, repr=False)

    async def initialize(self, cwd: Path = None) -> None:
        """
        创建并连接所有子模块。
        必须在第一次调用 agentic_loop 之前调用。
        """
        if cwd:
            self.config.cwd = cwd.resolve()
        else:
            self.config.cwd = self.config.cwd.resolve()

        # 创建 LLM 客户端
        self.llm_client = LLMClient(
            model=self.config.model,
            base_url=self.config.base_url,
            api_key=self.config.api_key,
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
        )

        # 创建 ContextManager，带工作目录信息的系统 prompt
        system_prompt = build_system_prompt(working_directory=str(self.config.cwd))
        self.context_manager = ContextManager(system_prompt=system_prompt)

        # 创建 ToolRegistry，注册所有可用工具
        self.tool_registry = ToolRegistry()
        self._register_tools()

    def _register_tools(self) -> None:
        """注册所有工具。新增工具在这里加一行即可。"""
        from src.file_tools import (
            ReadFileTool, WriteFileTool, EditTool,
            ListDirectoryTool, GlobTool
        )
        from src.network_tools import WebSearchTool, WebFetchTool

        cwd = self.config.cwd
        for tool in [
            ReadFileTool(cwd),
            WriteFileTool(cwd),
            EditTool(cwd),
            ListDirectoryTool(cwd),
            GlobTool(cwd),
            WebSearchTool(),
            WebFetchTool(),
        ]:
            self.tool_registry.register(tool)

    def increment_turn(self) -> int:
        """交互轮次 +1，更新 updated_at，返回新的轮次数。"""
        self.turn_count += 1
        self.updated_at = datetime.now()
        return self.turn_count

    async def close(self) -> None:
        """关闭 HTTP 客户端连接。程序退出前调用。"""
        if self.llm_client:
            await self.llm_client.close()

    def uptime(self) -> str:
        """返回会话运行时长的可读字符串。"""
        delta = datetime.now() - self.created_at
        total_seconds = int(delta.total_seconds())
        if total_seconds < 60:
            return f"{total_seconds}s"
        minutes = total_seconds // 60
        seconds = total_seconds % 60
        return f"{minutes}m {seconds}s"


print("Session 类定义完成")

In [ ]:
# 测试 Session 基础功能（不需要真实 LLM 连接）
import asyncio

session = Session(config=SessionConfig(model="gpt-oss:120b", cwd=Path(".")))
print(f"session_id: {session.session_id}")
print(f"created_at: {session.created_at.strftime('%H:%M:%S')}")
print(f"turn_count: {session.turn_count}")

# 模拟几次轮次
for _ in range(3):
    t = session.increment_turn()
print(f"increment x3 → turn_count: {session.turn_count}")
print(f"uptime: {session.uptime()}")

In [ ]:
# 测试完整初始化（需要 src/ 目录下有对应模块）
# 如果 src/ 模块不存在，这里会 ImportError，这是正常的——说明要先跑前面几章
import os

src_dir = Path("..") / "src"
has_src = src_dir.exists()
print(f"src/ 目录存在: {has_src}")

if has_src:
    modules = list(src_dir.glob("*.py"))
    print(f"可用模块: {[m.name for m in modules]}")
    
    needed = ["llm_client.py", "context_manager.py", "tool_framework.py"]
    missing = [m for m in needed if not (src_dir / m).exists()]
    if missing:
        print(f"缺少模块: {missing}，跳过初始化测试")
    else:
        # 尝试初始化
        try:
            test_session = Session()
            await test_session.initialize(cwd=Path("."))
            print(f"初始化成功")
            print(f"工具数量: {len(test_session.tool_registry.list_tools())}")
            await test_session.close()
        except Exception as e:
            print(f"初始化异常（可能是 Ollama 未运行）: {type(e).__name__}: {e}")
else:
    print("src/ 不存在，跳过初始化测试")

## 2. Rich 终端 UI

### 为什么要有 UI 层

原始的 `print()` 没有结构：工具调用的参数、错误信息、统计数据全部是平铺文本，
用户根本看不清楚发生了什么。

Rich 提供：
- **Panel** — 带边框的信息块，视觉上区分不同类型的输出
- **Syntax** — 代码高亮
- **Table** — 表格布局（工具参数用这个）
- **Text** — 带颜色的文本

### 几个渲染函数

| 函数 | 触发时机 | 颜色主题 |
|------|----------|----------|
| `print_welcome` | 启动时 | 蓝色 |
| `print_tool_call` | 工具调用开始 | 黄色 |
| `print_tool_result` | 工具执行完 | 绿色/红色 |
| `print_error` | 出错 | 红色 |
| `print_stats` | /stats 命令 | 默认 |

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.text import Text
from rich.table import Table
from rich.rule import Rule
from rich import box
import json

console = Console()


def print_welcome(session: Session) -> None:
    """启动时显示欢迎信息面板。"""
    content = Text()
    content.append("模型", style="bold")
    content.append(f"     {session.config.model}\n")
    content.append("工作目录", style="bold")
    content.append(f" {session.config.cwd}\n")
    content.append("会话 ID", style="bold")
    content.append(f"  {session.session_id[:8]}...\n\n")
    content.append("命令\n", style="bold")
    commands = [
        ("  /help",  "显示帮助"),
        ("  /exit",  "退出"),
        ("  /clear", "清空对话历史"),
        ("  /stats", "查看统计信息"),
        ("  /tools", "列出可用工具"),
        ("  /model <name>", "切换模型"),
    ]
    for cmd, desc in commands:
        content.append(f"{cmd}", style="cyan")
        content.append(f"  {desc}\n")

    console.print(Panel(
        content,
        title="[bold blue]AI Agent[/bold blue]",
        border_style="blue",
        padding=(1, 2),
    ))


def print_tool_call(name: str, params: dict) -> None:
    """工具调用开始时的面板：显示工具名和参数。"""
    table = Table(box=box.SIMPLE, show_header=False, padding=(0, 1))
    table.add_column("参数", style="dim", width=16)
    table.add_column("值")

    for key, val in params.items():
        # 长字符串截断显示，避免撑破面板
        val_str = str(val)
        if len(val_str) > 80:
            val_str = val_str[:77] + "..."
        table.add_row(key, val_str)

    console.print(Panel(
        table,
        title=f"[bold yellow]调用工具: {name}[/bold yellow]",
        border_style="yellow",
    ))


def print_tool_result(name: str, result_content: str, success: bool = True) -> None:
    """工具执行结果。成功绿色，失败红色。"""
    # 结果内容截断，超长的内容只显示前 500 字符
    display = result_content
    if len(display) > 500:
        display = display[:500] + f"\n... [共 {len(result_content)} 字符，已截断]"

    color = "green" if success else "red"
    label = "完成" if success else "失败"
    console.print(Panel(
        display,
        title=f"[bold {color}]{name}: {label}[/bold {color}]",
        border_style=color,
    ))


def print_error(msg: str) -> None:
    """红色错误提示。"""
    console.print(f"[bold red]错误:[/bold red] {msg}")


def print_stats(session: Session) -> None:
    """显示会话统计信息（/stats 命令触发）。"""
    table = Table(title="会话统计", box=box.ROUNDED)
    table.add_column("指标", style="cyan", width=20)
    table.add_column("值", justify="right")

    table.add_row("会话 ID", session.session_id[:16] + "...")
    table.add_row("模型", session.config.model)
    table.add_row("工作目录", str(session.config.cwd))
    table.add_row("交互轮次", str(session.turn_count))
    table.add_row("运行时长", session.uptime())
    table.add_row("创建时间", session.created_at.strftime("%Y-%m-%d %H:%M:%S"))

    if session.context_manager:
        ctx_stats = session.context_manager.stats()
        table.add_row("消息数量", str(ctx_stats.get("message_count", "-")))
        table.add_row("估计 token", str(ctx_stats.get("estimated_tokens", "-")))
        table.add_row(
            "总 token 用量",
            str(ctx_stats.get("total_usage", {}).get("total_tokens", "-"))
        )

    if session.tool_registry:
        table.add_row("可用工具数", str(len(session.tool_registry.list_tools())))

    console.print(table)


print("Rich UI 函数定义完成")

In [ ]:
# 演示欢迎面板
demo_session = Session(config=SessionConfig(
    model="gpt-oss:120b",
    cwd=Path(".").resolve(),
))
print_welcome(demo_session)

In [ ]:
# 演示工具调用面板
print_tool_call("read_file", {
    "path": "src/llm_client.py",
    "offset": 1,
    "limit": 50,
})

# 演示工具结果（成功）
print_tool_result(
    "read_file",
    "   1 │ from dataclasses import dataclass\n   2 │ from openai import AsyncOpenAI\n   3 │ ...",
    success=True
)

# 演示工具结果（失败）
print_tool_result(
    "write_file",
    "PermissionError: 路径 '../../etc/passwd' 在工作目录之外",
    success=False
)

# 演示错误提示
print_error("Ollama 连接失败，请确认 ollama serve 正在运行")

In [ ]:
# 演示统计面板
for _ in range(5):
    demo_session.increment_turn()
print_stats(demo_session)

## 3. AgentEvent 渲染

`agentic_loop` 是一个异步生成器，yield 各种事件（第 06 章定义）。
CLI 需要在事件流里实时渲染输出。

### 事件类型

```
AgentEvent
├── agent_start        ← 开始处理用户输入
├── agent_end          ← 处理完成（含最终回答）
├── text_delta         ← 流式文本片段（实时打印）
├── text_complete      ← 完整文本（流结束）
├── tool_call_start    ← 工具调用开始
├── tool_call_complete ← 工具调用完成
├── round_start        ← 新的 LLM 轮次开始
└── round_end          ← 轮次结束
```

`render_event` 把这些事件映射到 Rich 输出。

In [ ]:
from enum import Enum
from dataclasses import dataclass as event_dataclass
from typing import Any


class EventType(str, Enum):
    """Agent 事件类型（与 06_agentic_loop 保持一致）。"""
    AGENT_START = "agent_start"
    AGENT_END = "agent_end"
    TEXT_DELTA = "text_delta"
    TEXT_COMPLETE = "text_complete"
    TOOL_CALL_START = "tool_call_start"
    TOOL_CALL_COMPLETE = "tool_call_complete"
    ROUND_START = "round_start"
    ROUND_END = "round_end"
    ERROR = "error"


@event_dataclass
class AgentEvent:
    """Agent 运行时事件。"""
    type: EventType
    data: Any = None       # 事件携带的数据（具体结构取决于 type）


def render_event(event: AgentEvent) -> None:
    """
    把 AgentEvent 渲染成终端输出。
    
    text_delta 不换行（流式实时打印效果），
    其他事件各自用 Panel / Rule 分隔。
    """
    t = event.type

    if t == EventType.TEXT_DELTA:
        # 流式片段：不换行，end="" 实现打字机效果
        console.print(event.data or "", end="", highlight=False)

    elif t == EventType.TEXT_COMPLETE:
        # 完整文本打印完后换行
        console.print()  # 换行

    elif t == EventType.TOOL_CALL_START:
        console.print()  # 先换行，避免和流式文本混在一起
        data = event.data or {}
        print_tool_call(
            name=data.get("name", "unknown"),
            params=data.get("params", {}),
        )

    elif t == EventType.TOOL_CALL_COMPLETE:
        data = event.data or {}
        print_tool_result(
            name=data.get("name", "unknown"),
            result_content=data.get("content", ""),
            success=data.get("success", True),
        )

    elif t == EventType.ROUND_START:
        data = event.data or {}
        round_num = data.get("round", "?")
        console.print(Rule(f"[dim]第 {round_num} 轮[/dim]", style="dim"))

    elif t == EventType.AGENT_END:
        # agent_end 不再单独打印，文本已经通过 text_delta 流式显示
        pass

    elif t == EventType.ERROR:
        print_error(str(event.data or "未知错误"))

    # agent_start / round_end 不显示任何东西（静默事件）


print("render_event 定义完成")

In [ ]:
# 演示事件流渲染（模拟一次完整的 agentic loop 输出）
from typing import AsyncIterator

async def mock_event_stream() -> AsyncIterator[AgentEvent]:
    """模拟 agentic_loop 产生的事件序列。"""
    yield AgentEvent(EventType.AGENT_START)
    yield AgentEvent(EventType.ROUND_START, {"round": 1})

    # LLM 决定调用工具
    yield AgentEvent(EventType.TOOL_CALL_START, {
        "name": "read_file",
        "params": {"path": "src/llm_client.py", "limit": 20},
    })
    yield AgentEvent(EventType.TOOL_CALL_COMPLETE, {
        "name": "read_file",
        "content": "   1 │ from dataclasses import dataclass\n   2 │ ...",
        "success": True,
    })

    yield AgentEvent(EventType.ROUND_START, {"round": 2})

    # LLM 流式回答
    answer = "这个文件定义了 LLMClient 类，用于封装 Ollama 的 API 调用。"
    for char in answer:
        yield AgentEvent(EventType.TEXT_DELTA, char)
    yield AgentEvent(EventType.TEXT_COMPLETE)

    yield AgentEvent(EventType.AGENT_END)


# 运行演示
console.print(Rule("[bold]事件流渲染演示[/bold]"))
async for event in mock_event_stream():
    render_event(event)
console.print(Rule("[dim]演示结束[/dim]", style="dim"))

## 4. CLI 命令处理

`/` 开头的命令不发给 LLM，由本地函数直接处理。

| 命令 | 行为 |
|------|------|
| `/help` | 打印命令列表 |
| `/exit` | 退出（返回 False 让主循环结束） |
| `/clear` | 清空对话历史 |
| `/stats` | 显示统计面板 |
| `/tools` | 列出所有注册的工具 |
| `/model <name>` | 切换模型（需要重建 LLMClient） |

返回 `True` 表示继续运行，`False` 表示退出。

In [ ]:
def print_help() -> None:
    """打印命令帮助。"""
    table = Table(title="可用命令", box=box.SIMPLE)
    table.add_column("命令", style="cyan", width=20)
    table.add_column("说明")

    commands = [
        ("/help",       "显示这个帮助"),
        ("/exit",       "退出程序"),
        ("/clear",      "清空对话历史（保留系统 prompt）"),
        ("/stats",      "查看会话统计：轮次、token 用量、运行时长"),
        ("/tools",      "列出所有可用工具"),
        ("/model <名称>", "切换模型，例如: /model qwen2.5-coder:7b"),
    ]
    for cmd, desc in commands:
        table.add_row(cmd, desc)

    console.print(table)


def print_tools(session: Session) -> None:
    """列出所有注册工具的名称、类型和描述。"""
    if not session.tool_registry:
        print_error("Session 尚未初始化")
        return

    tools = session.tool_registry.list_tools()
    if not tools:
        console.print("[dim]没有注册任何工具[/dim]")
        return

    table = Table(title=f"可用工具（共 {len(tools)} 个）", box=box.ROUNDED)
    table.add_column("工具名", style="cyan", width=20)
    table.add_column("类型", width=10)
    table.add_column("描述")

    for t in tools:
        # 描述只取第一句话
        short_desc = t.description.split("。")[0].split(". ")[0]
        table.add_row(t.name, t.kind.value, short_desc)

    console.print(table)


def handle_slash_command(cmd: str, session: Session) -> bool:
    """
    处理 / 开头的命令。
    
    返回 True: 继续运行
    返回 False: 退出程序
    """
    parts = cmd.strip().split(maxsplit=1)
    command = parts[0].lower()
    arg = parts[1] if len(parts) > 1 else ""

    if command == "/help":
        print_help()

    elif command == "/exit" or command == "/quit":
        console.print("[dim]再见。[/dim]")
        return False  # 信号：退出

    elif command == "/clear":
        if session.context_manager:
            session.context_manager.clear_messages()
            console.print("[green]对话历史已清空（系统 prompt 保留）[/green]")
        else:
            print_error("Session 尚未初始化")

    elif command == "/stats":
        print_stats(session)

    elif command == "/tools":
        print_tools(session)

    elif command == "/model":
        if not arg:
            print_error("用法: /model <模型名>，例如: /model qwen2.5-coder:7b")
        else:
            session.config.model = arg
            if session.llm_client:
                session.llm_client.model = arg
            console.print(f"[green]模型已切换为: {arg}[/green]")
            console.print("[dim]注意：已有对话历史不受影响[/dim]")

    else:
        print_error(f"未知命令: {command}，输入 /help 查看可用命令")

    return True  # 继续运行


print("CLI 命令处理函数定义完成")

In [ ]:
# 测试 slash command 处理
test_session = Session(config=SessionConfig(model="gpt-oss:120b", cwd=Path(".")))

# 测试 /help
console.print(Rule("[bold]/help[/bold]"))
handle_slash_command("/help", test_session)

# 测试 /stats（未初始化时）
console.print(Rule("[bold]/stats（未初始化）[/bold]"))
handle_slash_command("/stats", test_session)

# 测试 /model 切换
console.print(Rule("[bold]/model 切换[/bold]"))
result = handle_slash_command("/model qwen2.5-coder:7b", test_session)
print(f"继续运行: {result}")
print(f"当前模型: {test_session.config.model}")

# 测试 /exit
console.print(Rule("[bold]/exit[/bold]"))
result = handle_slash_command("/exit", test_session)
print(f"继续运行: {result}")

In [ ]:
# 测试未知命令
handle_slash_command("/debug", test_session)

## 5. 交互式主循环

主循环的逻辑很简单：

```
打印欢迎 → 读用户输入 → 判断是否是 / 命令 → 交给 agentic_loop
                                                    ↑
                                            实时渲染事件
```

几个细节：
- `KeyboardInterrupt`（Ctrl+C）不退出，只提示用 `/exit`
- `EOFError` 在管道输入结束时触发，同样提示
- 空输入直接跳过
- `agentic_loop` 跑完后换行，避免下一个 prompt 紧接在输出后面

In [ ]:
from typing import AsyncIterator, Callable


async def run_interactive(
    session: Session,
    agentic_loop_fn: Callable = None,
) -> None:
    """
    交互式主循环。
    
    agentic_loop_fn: 异步生成器工厂，签名为:
        async def agentic_loop(user_input, context_manager, tool_registry,
                               llm_client, max_turns) -> AsyncIterator[AgentEvent]
    
    如果 agentic_loop_fn 为 None，则尝试从 src.agentic_loop 导入。
    """
    if agentic_loop_fn is None:
        try:
            from src.agentic_loop import agentic_loop as agentic_loop_fn
        except ImportError:
            print_error("找不到 src/agentic_loop.py，请先完成第 06 章")
            return

    print_welcome(session)

    while True:
        # 读用户输入
        try:
            user_input = input("你 > ").strip()
        except KeyboardInterrupt:
            # Ctrl+C：不退出，给提示
            console.print("\n[dim]按 Ctrl+C 不会退出，使用 /exit 退出[/dim]")
            continue
        except EOFError:
            # 管道结束（如 echo "..." | python agent.py）
            console.print("\n[dim]输入结束[/dim]")
            break

        if not user_input:
            continue  # 空输入跳过

        # slash 命令
        if user_input.startswith("/"):
            should_continue = handle_slash_command(user_input, session)
            if not should_continue:
                break
            continue

        # 正常对话
        turn = session.increment_turn()
        console.print(f"[dim]第 {turn} 轮[/dim]")

        try:
            async for event in agentic_loop_fn(
                user_input=user_input,
                context_manager=session.context_manager,
                tool_registry=session.tool_registry,
                llm_client=session.llm_client,
                max_turns=session.config.max_turns,
            ):
                render_event(event)
        except Exception as e:
            print_error(f"Agent 运行异常: {type(e).__name__}: {e}")

        console.print()  # 回答结束后空一行，视觉分隔


print("run_interactive 定义完成")

In [ ]:
# 测试：用 mock agentic_loop 验证主循环逻辑
# （不需要真实 Ollama 连接）

async def mock_agentic_loop(
    user_input: str,
    context_manager,
    tool_registry,
    llm_client,
    max_turns: int = 10,
) -> AsyncIterator[AgentEvent]:
    """模拟 agentic_loop，用于测试 CLI 渲染逻辑。"""
    yield AgentEvent(EventType.AGENT_START)
    yield AgentEvent(EventType.ROUND_START, {"round": 1})

    # 模拟工具调用
    if "文件" in user_input or "list" in user_input.lower():
        yield AgentEvent(EventType.TOOL_CALL_START, {
            "name": "list_directory",
            "params": {"path": "."},
        })
        yield AgentEvent(EventType.TOOL_CALL_COMPLETE, {
            "name": "list_directory",
            "content": "src/\ncourse/\nREADME.md",
            "success": True,
        })
        yield AgentEvent(EventType.ROUND_START, {"round": 2})

    # 模拟流式文本
    response = f"收到你的问题：{user_input}。这是来自 mock Agent 的回复。"
    for char in response:
        yield AgentEvent(EventType.TEXT_DELTA, char)
    yield AgentEvent(EventType.TEXT_COMPLETE)
    yield AgentEvent(EventType.AGENT_END)


# 模拟一次对话，不进入真实交互循环
test_session2 = Session(config=SessionConfig(
    model="gpt-oss:120b",
    cwd=Path(".").resolve(),
))

console.print(Rule("[bold]单次对话测试[/bold]"))
console.print(f"[dim]输入: 列出工作目录的文件[/dim]\n")

async for event in mock_agentic_loop(
    user_input="列出工作目录的文件",
    context_manager=None,
    tool_registry=None,
    llm_client=None,
):
    render_event(event)

console.print(Rule("[dim]对话结束[/dim]", style="dim"))

## 6. Click CLI 入口

Click 负责解析命令行参数。两种运行模式：

1. **交互模式**（默认）：进入 `run_interactive()` 循环
2. **一次性模式**（`--prompt`）：跑一次 agentic_loop 就退出，适合脚本调用

```bash
# 交互模式
python -m src.cli

# 一次性模式
python -m src.cli --prompt "读 README.md 然后总结一下"

# 指定模型和工作目录
python -m src.cli --model qwen2.5-coder:7b --cwd /path/to/project
```

In [ ]:
import asyncio
import click


async def run_once(
    session: Session,
    prompt: str,
    agentic_loop_fn: Callable = None,
) -> None:
    """
    一次性模式：执行单个 prompt 后退出。
    --prompt 参数触发。
    """
    if agentic_loop_fn is None:
        try:
            from src.agentic_loop import agentic_loop as agentic_loop_fn
        except ImportError:
            print_error("找不到 src/agentic_loop.py，请先完成第 06 章")
            return

    console.print(f"[dim]提问: {prompt}[/dim]")
    console.print(Rule(style="dim"))

    session.increment_turn()
    async for event in agentic_loop_fn(
        user_input=prompt,
        context_manager=session.context_manager,
        tool_registry=session.tool_registry,
        llm_client=session.llm_client,
        max_turns=session.config.max_turns,
    ):
        render_event(event)

    console.print()


async def init_and_run(
    session: Session,
    prompt: str | None,
    model: str,
    cwd: str,
    max_turns: int,
) -> None:
    """异步入口：初始化 Session，然后进入对应运行模式。"""
    session.config.model = model
    session.config.max_turns = max_turns

    try:
        await session.initialize(cwd=Path(cwd))
    except Exception as e:
        print_error(f"Session 初始化失败: {e}")
        print_error("请确认 Ollama 正在运行: ollama serve")
        return

    try:
        if prompt:
            await run_once(session, prompt)
        else:
            await run_interactive(session)
    finally:
        await session.close()


# Click 命令定义
# 注意：在 Notebook 里不能直接调用 cli()，需要用 asyncio.run() 或 CliRunner
@click.command()
@click.option("--prompt", "-p", default=None, help="一次性问题，不进入交互模式")
@click.option("--model", "-m", default="gpt-oss:120b", show_default=True,
              help="使用的 Ollama 模型")
@click.option("--cwd", default=".", show_default=True,
              help="Agent 的工作目录")
@click.option("--max-turns", default=10, show_default=True,
              help="单次对话最多运行几轮 agentic loop")
def cli(prompt: str, model: str, cwd: str, max_turns: int):
    """AI Agent 命令行界面。"""
    session = Session()
    asyncio.run(init_and_run(session, prompt, model, cwd, max_turns))


print("Click CLI 入口定义完成")
print()
print("使用方式:")
print("  python -m src.cli                           # 交互模式")
print("  python -m src.cli -p '帮我看一下有哪些 .py 文件'  # 一次性模式")
print("  python -m src.cli -m qwen2.5-coder:7b       # 指定模型")

In [ ]:
# 用 Click 的 CliRunner 测试 CLI（不依赖真实终端）
from click.testing import CliRunner

runner = CliRunner()

# 测试 --help
result = runner.invoke(cli, ["--help"])
print(result.output)

## 7. 端到端测试：真实 Ollama 连接

下面这个 cell 需要 Ollama 正在运行。测试流程：

1. 创建 Session，初始化所有子模块
2. 用 mock agentic_loop 跑一轮对话（验证 Session + CLI 集成）
3. 模拟几个 slash 命令，确认输出正确

如果 Ollama 没有运行，初始化会失败并打印错误，不会崩溃。

In [ ]:
# 测试 Session 初始化
integration_session = Session(config=SessionConfig(
    model="gpt-oss:120b",
    cwd=Path(".").resolve(),
))

try:
    await integration_session.initialize()
    print(f"初始化成功")
    print(f"模型: {integration_session.config.model}")
    print(f"工作目录: {integration_session.config.cwd}")
    print(f"工具数量: {len(integration_session.tool_registry.list_tools())}")
    initialized = True
except Exception as e:
    print(f"初始化失败（可能是 Ollama 未运行）: {type(e).__name__}: {e}")
    initialized = False

In [ ]:
# 测试 /tools 命令（需要初始化成功）
if initialized:
    console.print(Rule("[bold]/tools 命令测试[/bold]"))
    handle_slash_command("/tools", integration_session)
else:
    print("跳过（Session 未初始化）")

In [ ]:
# 测试 /stats 命令（需要初始化成功）
if initialized:
    # 模拟几次交互
    for _ in range(3):
        integration_session.increment_turn()
    
    console.print(Rule("[bold]/stats 命令测试[/bold]"))
    handle_slash_command("/stats", integration_session)
else:
    print("跳过（Session 未初始化）")

In [ ]:
# 用 mock agentic_loop 跑一轮完整对话
if initialized:
    console.print(Rule("[bold]单轮对话测试[/bold]"))
    
    user_msg = "列出工作目录下的 Python 文件"
    console.print(f"你 > {user_msg}")
    console.print()
    
    integration_session.increment_turn()
    async for event in mock_agentic_loop(
        user_input=user_msg,
        context_manager=integration_session.context_manager,
        tool_registry=integration_session.tool_registry,
        llm_client=integration_session.llm_client,
    ):
        render_event(event)
    
    console.print()
    console.print(Rule("[dim]对话结束[/dim]", style="dim"))
else:
    print("跳过（Session 未初始化）")

In [ ]:
# 如果 Ollama 在线，跑一次真实的 LLM 请求（不用工具）
if initialized:
    print("测试真实 LLM 连接...")
    try:
        reply, usage = await integration_session.llm_client.chat_completion(
            messages=[
                {"role": "user", "content": "用一句话说明 Session 对象在 Agent 架构中的作用"}
            ]
        )
        console.print(Panel(
            reply,
            title="[green]LLM 回答[/green]",
            border_style="green",
        ))
        print(f"token 用量: {usage.total_tokens}")
    except Exception as e:
        print(f"LLM 调用失败: {type(e).__name__}: {e}")

    await integration_session.close()
else:
    print("跳过（Session 未初始化）")

## 8. 保存到 src/

三个文件：
- `src/session.py` — Session 类
- `src/tui.py` — Rich UI 函数和事件渲染
- `src/cli.py` — Click 入口

In [ ]:
import os

src_dir = Path("..") / "src"
src_dir.mkdir(exist_ok=True)

# ──────────────────────────────
# src/session.py
# ──────────────────────────────
session_py = '''
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from uuid import uuid4

from src.llm_client import LLMClient
from src.context_manager import ContextManager, build_system_prompt
from src.tool_framework import ToolRegistry


@dataclass
class SessionConfig:
    """Session 运行时参数。"""
    model: str = "gpt-oss:120b"
    base_url: str = "http://localhost:11434/v1"
    api_key: str = "ollama"
    max_turns: int = 10
    temperature: float = 0.7
    max_tokens: int = 4096
    cwd: Path = field(default_factory=lambda: Path("."))


@dataclass
class Session:
    """一次 Agent 会话的完整上下文容器。"""
    session_id: str = field(default_factory=lambda: str(uuid4()))
    created_at: datetime = field(default_factory=datetime.now)
    updated_at: datetime = field(default_factory=datetime.now)
    turn_count: int = 0
    config: SessionConfig = field(default_factory=SessionConfig)
    context_manager: ContextManager = field(default=None, repr=False)
    tool_registry: ToolRegistry = field(default=None, repr=False)
    llm_client: LLMClient = field(default=None, repr=False)

    async def initialize(self, cwd: Path = None) -> None:
        if cwd:
            self.config.cwd = cwd.resolve()
        else:
            self.config.cwd = self.config.cwd.resolve()

        self.llm_client = LLMClient(
            model=self.config.model,
            base_url=self.config.base_url,
            api_key=self.config.api_key,
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
        )
        system_prompt = build_system_prompt(working_directory=str(self.config.cwd))
        self.context_manager = ContextManager(system_prompt=system_prompt)
        self.tool_registry = ToolRegistry()
        self._register_tools()

    def _register_tools(self) -> None:
        from src.file_tools import (
            ReadFileTool, WriteFileTool, EditTool,
            ListDirectoryTool, GlobTool
        )
        from src.network_tools import WebSearchTool, WebFetchTool
        cwd = self.config.cwd
        for tool in [
            ReadFileTool(cwd),
            WriteFileTool(cwd),
            EditTool(cwd),
            ListDirectoryTool(cwd),
            GlobTool(cwd),
            WebSearchTool(),
            WebFetchTool(),
        ]:
            self.tool_registry.register(tool)

    def increment_turn(self) -> int:
        self.turn_count += 1
        self.updated_at = datetime.now()
        return self.turn_count

    async def close(self) -> None:
        if self.llm_client:
            await self.llm_client.close()

    def uptime(self) -> str:
        delta = datetime.now() - self.created_at
        total_seconds = int(delta.total_seconds())
        if total_seconds < 60:
            return f"{total_seconds}s"
        minutes = total_seconds // 60
        seconds = total_seconds % 60
        return f"{minutes}m {seconds}s"
'''

(src_dir / "session.py").write_text(session_py.strip())
print("写入 src/session.py")

In [ ]:
# ──────────────────────────────
# src/tui.py
# ──────────────────────────────
tui_py = '''
from enum import Enum
from dataclasses import dataclass
from typing import Any

from rich.console import Console
from rich.panel import Panel
from rich.text import Text
from rich.table import Table
from rich.rule import Rule
from rich import box

console = Console()


class EventType(str, Enum):
    """Agent 事件类型。"""
    AGENT_START = "agent_start"
    AGENT_END = "agent_end"
    TEXT_DELTA = "text_delta"
    TEXT_COMPLETE = "text_complete"
    TOOL_CALL_START = "tool_call_start"
    TOOL_CALL_COMPLETE = "tool_call_complete"
    ROUND_START = "round_start"
    ROUND_END = "round_end"
    ERROR = "error"


@dataclass
class AgentEvent:
    """Agent 运行时事件。"""
    type: EventType
    data: Any = None


def print_welcome(session) -> None:
    content = Text()
    content.append("模型", style="bold")
    content.append(f"     {session.config.model}\\n")
    content.append("工作目录", style="bold")
    content.append(f" {session.config.cwd}\\n")
    content.append("会话 ID", style="bold")
    content.append(f"  {session.session_id[:8]}...\\n\\n")
    content.append("命令\\n", style="bold")
    for cmd, desc in [
        ("  /help", "显示帮助"),
        ("  /exit", "退出"),
        ("  /clear", "清空对话历史"),
        ("  /stats", "查看统计信息"),
        ("  /tools", "列出可用工具"),
        ("  /model <name>", "切换模型"),
    ]:
        content.append(cmd, style="cyan")
        content.append(f"  {desc}\\n")
    console.print(Panel(content, title="[bold blue]AI Agent[/bold blue]",
                        border_style="blue", padding=(1, 2)))


def print_tool_call(name: str, params: dict) -> None:
    table = Table(box=box.SIMPLE, show_header=False, padding=(0, 1))
    table.add_column("参数", style="dim", width=16)
    table.add_column("值")
    for key, val in params.items():
        val_str = str(val)
        if len(val_str) > 80:
            val_str = val_str[:77] + "..."
        table.add_row(key, val_str)
    console.print(Panel(table,
                        title=f"[bold yellow]调用工具: {name}[/bold yellow]",
                        border_style="yellow"))


def print_tool_result(name: str, result_content: str, success: bool = True) -> None:
    display = result_content
    if len(display) > 500:
        display = display[:500] + f"\\n... [共 {len(result_content)} 字符，已截断]"
    color = "green" if success else "red"
    label = "完成" if success else "失败"
    console.print(Panel(display,
                        title=f"[bold {color}]{name}: {label}[/bold {color}]",
                        border_style=color))


def print_error(msg: str) -> None:
    console.print(f"[bold red]错误:[/bold red] {msg}")


def print_stats(session) -> None:
    table = Table(title="会话统计", box=box.ROUNDED)
    table.add_column("指标", style="cyan", width=20)
    table.add_column("值", justify="right")
    table.add_row("会话 ID", session.session_id[:16] + "...")
    table.add_row("模型", session.config.model)
    table.add_row("工作目录", str(session.config.cwd))
    table.add_row("交互轮次", str(session.turn_count))
    table.add_row("运行时长", session.uptime())
    if session.context_manager:
        ctx_stats = session.context_manager.stats()
        table.add_row("消息数量", str(ctx_stats.get("message_count", "-")))
        table.add_row("估计 token", str(ctx_stats.get("estimated_tokens", "-")))
    if session.tool_registry:
        table.add_row("可用工具数", str(len(session.tool_registry.list_tools())))
    console.print(table)


def render_event(event: AgentEvent) -> None:
    """把 AgentEvent 渲染成终端输出。"""
    t = event.type
    if t == EventType.TEXT_DELTA:
        console.print(event.data or "", end="", highlight=False)
    elif t == EventType.TEXT_COMPLETE:
        console.print()
    elif t == EventType.TOOL_CALL_START:
        console.print()
        data = event.data or {}
        print_tool_call(name=data.get("name", "unknown"), params=data.get("params", {}))
    elif t == EventType.TOOL_CALL_COMPLETE:
        data = event.data or {}
        print_tool_result(
            name=data.get("name", "unknown"),
            result_content=data.get("content", ""),
            success=data.get("success", True),
        )
    elif t == EventType.ROUND_START:
        data = event.data or {}
        console.print(Rule(f"[dim]第 {data.get(\'round\', \'?\')} 轮[/dim]", style="dim"))
    elif t == EventType.ERROR:
        print_error(str(event.data or "未知错误"))
'''

(src_dir / "tui.py").write_text(tui_py.strip())
print("写入 src/tui.py")

In [ ]:
# ──────────────────────────────
# src/cli.py
# ──────────────────────────────
cli_py = '''
import asyncio
from pathlib import Path
from typing import Callable

import click

from src.session import Session, SessionConfig
from src.tui import console, print_welcome, print_error, render_event
from src.tui import AgentEvent, EventType


def handle_slash_command(cmd: str, session) -> bool:
    """处理 / 开头的命令，返回 True 继续，False 退出。"""
    from rich.table import Table
    from rich import box
    from rich.rule import Rule

    parts = cmd.strip().split(maxsplit=1)
    command = parts[0].lower()
    arg = parts[1] if len(parts) > 1 else ""

    if command == "/help":
        table = Table(title="可用命令", box=box.SIMPLE)
        table.add_column("命令", style="cyan", width=20)
        table.add_column("说明")
        for c, d in [
            ("/help", "显示帮助"),
            ("/exit", "退出程序"),
            ("/clear", "清空对话历史"),
            ("/stats", "查看会话统计"),
            ("/tools", "列出可用工具"),
            ("/model <名称>", "切换模型"),
        ]:
            table.add_row(c, d)
        console.print(table)

    elif command in ("/exit", "/quit"):
        console.print("[dim]再见。[/dim]")
        return False

    elif command == "/clear":
        if session.context_manager:
            session.context_manager.clear_messages()
            console.print("[green]对话历史已清空[/green]")
        else:
            print_error("Session 尚未初始化")

    elif command == "/stats":
        from src.tui import print_stats
        print_stats(session)

    elif command == "/tools":
        if not session.tool_registry:
            print_error("Session 尚未初始化")
        else:
            tools = session.tool_registry.list_tools()
            table = Table(title=f"可用工具（共 {len(tools)} 个）", box=box.ROUNDED)
            table.add_column("工具名", style="cyan", width=20)
            table.add_column("类型", width=10)
            table.add_column("描述")
            for t in tools:
                short = t.description.split("。")[0][:50]
                table.add_row(t.name, t.kind.value, short)
            console.print(table)

    elif command == "/model":
        if not arg:
            print_error("用法: /model <模型名>")
        else:
            session.config.model = arg
            if session.llm_client:
                session.llm_client.model = arg
            console.print(f"[green]模型已切换为: {arg}[/green]")

    else:
        print_error(f"未知命令: {command}，输入 /help 查看")

    return True


async def run_interactive(session, agentic_loop_fn: Callable = None) -> None:
    """交互式主循环。"""
    if agentic_loop_fn is None:
        from src.agentic_loop import agentic_loop as agentic_loop_fn

    print_welcome(session)

    while True:
        try:
            user_input = input("你 > ").strip()
        except KeyboardInterrupt:
            console.print("\\n[dim]使用 /exit 退出[/dim]")
            continue
        except EOFError:
            break

        if not user_input:
            continue

        if user_input.startswith("/"):
            if not handle_slash_command(user_input, session):
                break
            continue

        turn = session.increment_turn()
        console.print(f"[dim]第 {turn} 轮[/dim]")
        try:
            async for event in agentic_loop_fn(
                user_input=user_input,
                context_manager=session.context_manager,
                tool_registry=session.tool_registry,
                llm_client=session.llm_client,
                max_turns=session.config.max_turns,
            ):
                render_event(event)
        except Exception as e:
            print_error(f"Agent 运行异常: {type(e).__name__}: {e}")
        console.print()


async def run_once(session, prompt: str, agentic_loop_fn: Callable = None) -> None:
    """一次性模式：执行 prompt 后退出。"""
    if agentic_loop_fn is None:
        from src.agentic_loop import agentic_loop as agentic_loop_fn

    from rich.rule import Rule
    console.print(f"[dim]提问: {prompt}[/dim]")
    console.print(Rule(style="dim"))
    session.increment_turn()
    async for event in agentic_loop_fn(
        user_input=prompt,
        context_manager=session.context_manager,
        tool_registry=session.tool_registry,
        llm_client=session.llm_client,
        max_turns=session.config.max_turns,
    ):
        render_event(event)
    console.print()


async def init_and_run(
    session,
    prompt: str | None,
    model: str,
    cwd: str,
    max_turns: int,
) -> None:
    session.config.model = model
    session.config.max_turns = max_turns
    try:
        await session.initialize(cwd=Path(cwd))
    except Exception as e:
        print_error(f"Session 初始化失败: {e}")
        print_error("请确认 Ollama 正在运行: ollama serve")
        return
    try:
        if prompt:
            await run_once(session, prompt)
        else:
            await run_interactive(session)
    finally:
        await session.close()


@click.command()
@click.option("--prompt", "-p", default=None, help="一次性问题，不进入交互模式")
@click.option("--model", "-m", default="gpt-oss:120b", show_default=True)
@click.option("--cwd", default=".", show_default=True, help="Agent 工作目录")
@click.option("--max-turns", default=10, show_default=True)
def main(prompt: str, model: str, cwd: str, max_turns: int):
    """AI Agent 命令行界面。"""
    session = Session()
    asyncio.run(init_and_run(session, prompt, model, cwd, max_turns))


if __name__ == "__main__":
    main()
'''

(src_dir / "cli.py").write_text(cli_py.strip())
print("写入 src/cli.py")
print()
print("完成：")
for fname in ["session.py", "tui.py", "cli.py"]:
    p = src_dir / fname
    if p.exists():
        print(f"  {p}  ({p.stat().st_size} bytes)")

## 9. 小结

| 模块 | 作用 | 文件 |
|------|------|------|
| `Session` | 持有 LLMClient、ContextManager、ToolRegistry 和会话元数据 | `src/session.py` |
| `SessionConfig` | CLI 参数的容器，统一传给 Session 子模块 | `src/session.py` |
| `print_welcome` | 启动时显示模型、目录、可用命令 | `src/tui.py` |
| `print_tool_call` | 工具调用开始时显示工具名和参数表格 | `src/tui.py` |
| `print_tool_result` | 成功绿色/失败红色 | `src/tui.py` |
| `print_stats` | /stats 命令输出统计信息 | `src/tui.py` |
| `render_event` | 把 AgentEvent 流映射到 Rich 输出 | `src/tui.py` |
| `handle_slash_command` | /help /exit /clear /stats /tools /model | `src/cli.py` |
| `run_interactive` | 交互式主循环，实时渲染事件 | `src/cli.py` |
| `run_once` | 一次性模式（--prompt 参数） | `src/cli.py` |
| `main` (Click) | 命令行入口，解析参数，进入对应模式 | `src/cli.py` |

**设计要点**：Session 的 `initialize()` 是 async 的，Click 的 `main()` 用 `asyncio.run()` 桥接同步入口和异步逻辑。UI 层（tui.py）和业务逻辑（session.py、cli.py）分开，`render_event` 是唯一的 UI 入口，agentic_loop 不直接 print。

下一章（08）做**上下文压缩**：当对话历史超过 token 窗口的 80%，自动调用 LLM 把历史压缩成摘要，让 Agent 可以无限期运行而不爆 context。